# Stream Processing Pipeline — Mercado de Criptomoedas
## MBA em Engenharia de Dados | Disciplina: Stream Processing Pipelines
**Prof. Rafael Tsuji Matsuyama** | Trabalho Final

---

### Visão Geral do Pipeline

Este notebook implementa um pipeline de dados de streaming que consome cotações de criptomoedas em tempo real da **CoinGecko API** (API pública, gratuita, sem autenticação), processa os dados com **Apache Spark Structured Streaming** no Databricks, e persiste o resultado em formato **Parquet** (Delta Lake).

```
CoinGecko API (JSON)
       │
       ▼
  Ingestão via polling (readStream)
       │
       ▼
  Validação de Schema + Filtro de qualidade
       │
       ▼
  Agregação: média, máx, mín por moeda
       │
       ▼
  Window Functions: janela deslizante de 1h
       │
       ▼
  Output: Delta Lake (Parquet) no DBFS
```

**Tecnologias utilizadas:**
- Apache Spark Structured Streaming (Databricks Runtime 13+)
- CoinGecko Public API v3
- Delta Lake (formato Parquet)
- Python 3.10+

**Fonte de dados:** [CoinGecko API](https://docs.coingecko.com) — maior agregador independente de dados de criptomoedas do mundo, com mais de 18.000 moedas e 1.000+ exchanges integradas. Tier gratuito: 30 chamadas/minuto, sem necessidade de API key.


---
## Parte 1 — Instalação de dependências e imports
Instalamos a biblioteca `requests` para as chamadas HTTP à API.
No Databricks, o `pyspark` já está disponível no ambiente.

In [0]:
# Instalação de dependências externas
# No Databricks, execute em um cluster com DBR 13.x ou superior
%pip install requests --quiet

# Imports padrão
import requests
import json
import time
import os
from datetime import datetime
from pathlib import Path

# Imports PySpark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType, LongType, TimestampType
)
from pyspark.sql.window import Window

print("✅ Dependências carregadas com sucesso")

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
✅ Dependências carregadas com sucesso


In [0]:
# Validação de versão
spark.version

'4.1.0'

---
## Parte 2 — Configuração do ambiente Databricks e diretórios

Configuramos os caminhos no DBFS (Databricks File System) para:
- `landing_path`: onde os arquivos JSON brutos chegam (zona de pouso)
- `output_path`: onde o Parquet processado é salvo (zona de consumo)
- `checkpoint_path`: estado do stream (obrigatório para retomada exata em caso de falha)

In [0]:
# ── Configuração do SparkSession ──────────────────────────────────────────────
# No Databricks, o SparkSession já existe como 'spark'.
# Este bloco garante compatibilidade com execução local também.

try:
    spark  # Databricks: variável já existe no contexto
    DATABRICKS = True
    print("✅ Executando no Databricks")
except NameError:
    spark = (
        SparkSession.builder
        .appName("StreamPipeline_CoinGecko")
        .config("spark.sql.streaming.checkpointLocation", "/tmp/checkpoint")
        .getOrCreate()
    )
    DATABRICKS = False
    print("✅ Executando localmente")

# ── Definição dos caminhos no DBFS ────────────────────────────────────────────
# BASE_PATH       = "/Volumes/workspace/default/crypto_pipeline"   # raiz do projeto (Unity Catalog Volume)
BASE_PATH       = "/tmp/crypto_pipeline" # /tmp funciona em qualquer cluster Databricks sem configuração adicional
LANDING_PATH    = f"{BASE_PATH}/landing"         # JSONs brutos (zona de pouso)
OUTPUT_PATH     = f"{BASE_PATH}/output"          # Parquet processado
CHECKPOINT_PATH = f"{BASE_PATH}/checkpoint"      # Estado do stream
AGGS_PATH       = f"{BASE_PATH}/aggregations"    # Resultados de agregação

# Cria os diretórios se não existirem
for path in [LANDING_PATH, OUTPUT_PATH, CHECKPOINT_PATH, AGGS_PATH]:
    os.makedirs(path, exist_ok=True)

print(f"\n📁 Diretórios configurados:")
print(f"   Landing (JSON bruto):  {LANDING_PATH}")
print(f"   Output  (Parquet):     {OUTPUT_PATH}")
print(f"   Checkpoint (estado):   {CHECKPOINT_PATH}")
print(f"   Agregações:            {AGGS_PATH}")

✅ Executando no Databricks

📁 Diretórios configurados:
   Landing (JSON bruto):  /tmp/crypto_pipeline/landing
   Output  (Parquet):     /tmp/crypto_pipeline/output
   Checkpoint (estado):   /tmp/crypto_pipeline/checkpoint
   Agregações:            /tmp/crypto_pipeline/aggregations


---
## Parte 3 — Definição do Schema Estruturado

Antes de qualquer dado entrar no pipeline, definimos explicitamente o schema esperado. Isso é chamado de **schema-on-read** — o Spark valida cada evento contra esse esquema na entrada.

Campos esperados da CoinGecko API (`/coins/markets`):
- `id`: identificador único da moeda (ex: `bitcoin`)
- `symbol`: símbolo de mercado (ex: `btc`)
- `name`: nome completo (ex: `Bitcoin`)
- `current_price`: preço atual em USD
- `market_cap`: capitalização de mercado em USD
- `total_volume`: volume de negociação nas últimas 24h
- `price_change_percentage_24h`: variação percentual em 24h
- `last_updated`: timestamp da última atualização

In [0]:
# ── Schema estruturado dos eventos de entrada ──────────────────────
# Definir schema explicitamente é uma boa prática em engenharia de dados:
# - Garante que campos obrigatórios estejam presentes
# - Evita inferência automática (cara em streaming)
# - Rejeita ou sinaliza eventos malformados

COIN_SCHEMA = StructType([
    StructField("id",                          StringType(),  nullable=False),  # obrigatório
    StructField("symbol",                      StringType(),  nullable=False),  # obrigatório
    StructField("name",                        StringType(),  nullable=False),  # obrigatório
    StructField("current_price",               DoubleType(),  nullable=True),
    StructField("market_cap",                  DoubleType(),  nullable=True),
    StructField("market_cap_rank",             LongType(),    nullable=True),
    StructField("total_volume",                DoubleType(),  nullable=True),
    StructField("high_24h",                    DoubleType(),  nullable=True),
    StructField("low_24h",                     DoubleType(),  nullable=True),
    StructField("price_change_percentage_24h", DoubleType(),  nullable=True),
    StructField("last_updated",                StringType(),  nullable=True),
])

# Campos obrigatórios — eventos sem estes campos serão descartados
REQUIRED_FIELDS = ["id", "symbol", "name", "current_price"]

print("✅ Schema definido com", len(COIN_SCHEMA.fields), "campos")
print("\nCampos obrigatórios:", REQUIRED_FIELDS)
print("\nSchema completo:")
for field in COIN_SCHEMA.fields:
    obrig = "[OBRIGATÓRIO]" if not field.nullable else "[opcional]    "
    print(f"  {obrig}  {field.name:<40} {str(field.dataType):<20}")

✅ Schema definido com 11 campos

Campos obrigatórios: ['id', 'symbol', 'name', 'current_price']

Schema completo:
  [OBRIGATÓRIO]  id                                       StringType()        
  [OBRIGATÓRIO]  symbol                                   StringType()        
  [OBRIGATÓRIO]  name                                     StringType()        
  [opcional]      current_price                            DoubleType()        
  [opcional]      market_cap                               DoubleType()        
  [opcional]      market_cap_rank                          LongType()          
  [opcional]      total_volume                             DoubleType()        
  [opcional]      high_24h                                 DoubleType()        
  [opcional]      low_24h                                  DoubleType()        
  [opcional]      price_change_percentage_24h              DoubleType()        
  [opcional]      last_updated                             StringType()        


---
## Parte 4 — Função de ingestão: CoinGecko API → JSON landing

Implementamos o **produtor de dados**: uma função que chama a CoinGecko API e persiste cada resposta como um arquivo JSON no diretório de landing. O Spark Structured Streaming monitora esse diretório continuamente e processa novos arquivos à medida que chegam — este é o padrão **file-based streaming source**.

**Por que polling e não WebSocket?**
A CoinGecko API gratuita é REST/JSON — não oferece WebSocket nativo no tier free. O padrão de polling (pull a cada N segundos) é perfeitamente válido para streaming de dados de mercado, onde a janela de atualização é de 1 a 5 minutos. Para o trabalho, executaremos o produtor em um loop antes de iniciar o pipeline.

In [0]:
# ── Configuração da CoinGecko API ─────────────────────────────────────────────

COINGECKO_URL = "https://api.coingecko.com/api/v3/coins/markets"

# Moedas monitoradas — top 10 por capitalização de mercado
COINS_TO_TRACK = [
    "bitcoin", "ethereum", "tether", "binancecoin", "solana",
    "ripple", "usd-coin", "staked-ether", "dogecoin", "cardano"
]

API_PARAMS = {
    "vs_currency": "usd",
    "ids": ",".join(COINS_TO_TRACK),
    "order": "market_cap_desc",
    "per_page": 10,
    "page": 1,
    "sparkline": False,
    "price_change_percentage": "24h"
}

def fetch_coin_data() -> list[dict]:
    """
    Chama a CoinGecko API e retorna a lista de moedas com seus dados de mercado.
    Retorna lista vazia em caso de erro (pipeline continua sem interrupção).
    """
    try:
        headers = {"accept": "application/json"}
        response = requests.get(COINGECKO_URL, params=API_PARAMS, headers=headers, timeout=10)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.HTTPError as e:
        # 429 = rate limit atingido — aguardar e tentar novamente
        if e.response.status_code == 429:
            print(f"⚠️  Rate limit (429) — aguardando 60s...")
            time.sleep(60)
        else:
            print(f"❌ Erro HTTP {e.response.status_code}: {e}")
        return []
    except Exception as e:
        print(f"❌ Erro na chamada à API: {e}")
        return []


def ingest_batch(batch_num: int) -> str:
    """
    Busca dados da API, adiciona metadados de ingestão e salva no landing.
    Retorna o caminho do arquivo salvo.
    """
    coins = fetch_coin_data()
    if not coins:
        return None

    # Adiciona timestamp de ingestão a cada registro
    ingestion_ts = datetime.utcnow().isoformat()
    for coin in coins:
        coin["ingestion_timestamp"] = ingestion_ts
        # Garante que campos numéricos existam (validação mínima antes de salvar)
        coin["current_price"] = coin.get("current_price") or 0.0
        coin["market_cap"]    = coin.get("market_cap")    or 0.0
        coin["total_volume"]  = coin.get("total_volume")  or 0.0

    filename = f"{LANDING_PATH}/batch_{batch_num:04d}_{int(time.time())}.json"
    with open(filename, "w") as f:
        json.dump(coins, f)

    print(f"📥 Batch {batch_num:04d} salvo — {len(coins)} moedas — {ingestion_ts}")
    return filename


# ── Teste de conectividade com a API ──────────────────────────────────────────
print("🔌 Testando conexão com a CoinGecko API...")
test_data = fetch_coin_data()
if test_data:
    print(f"✅ API respondeu com {len(test_data)} moedas")
    print(f"\nExemplo de registro (bitcoin):")
    btc = next((c for c in test_data if c["id"] == "bitcoin"), test_data[0])
    print(f"  id:             {btc.get('id')}")
    print(f"  name:           {btc.get('name')}")
    print(f"  current_price:  USD {btc.get('current_price'):,.2f}")
    print(f"  market_cap:     USD {btc.get('market_cap'):,.0f}")
    print(f"  24h change:     {btc.get('price_change_percentage_24h'):+.2f}%")
    print(f"  last_updated:   {btc.get('last_updated')}")
else:
    print("❌ Falha na conexão com a API")

🔌 Testando conexão com a CoinGecko API...
✅ API respondeu com 10 moedas

Exemplo de registro (bitcoin):
  id:             bitcoin
  name:           Bitcoin
  current_price:  USD 78,725.00
  market_cap:     USD 1,576,177,403,500
  24h change:     +0.82%
  last_updated:   2026-05-02T23:17:11.489Z


---
## Parte 5 — Geração de dados para o landing zone

Executamos o produtor em loop para simular a chegada contínua de dados. Cada iteração chama a API, adiciona timestamp de ingestão e salva um arquivo JSON no diretório de landing.

**Parâmetros:**
- `NUM_BATCHES = 5`: gera 5 snapshots com intervalo de 20 segundos entre cada um
- Total de eventos: ~50 registros (10 moedas × 5 batches)
- Tempo total: ~100 segundos

In [0]:
# ── Produtor de dados: gera batches no landing zone ───────────────────────────
# Ajuste NUM_BATCHES conforme necessário:
#   5  batches → demonstração rápida (~100s)
#  20  batches → melhor visualização das janelas temporais (~7min)

NUM_BATCHES      = 5    # número de chamadas à API
POLL_INTERVAL    = 20   # segundos entre cada chamada (respeitando rate limit)

print(f"🚀 Iniciando ingestão — {NUM_BATCHES} batches com intervalo de {POLL_INTERVAL}s")
print(f"   Tempo total estimado: {NUM_BATCHES * POLL_INTERVAL}s\n")

files_created = []
for i in range(1, NUM_BATCHES + 1):
    filepath = ingest_batch(i)
    if filepath:
        files_created.append(filepath)
    if i < NUM_BATCHES:
        time.sleep(POLL_INTERVAL)

print(f"\n✅ Ingestão concluída — {len(files_created)} arquivos criados no landing zone")
print(f"📁 Diretório: {LANDING_PATH}")

🚀 Iniciando ingestão — 5 batches com intervalo de 20s
   Tempo total estimado: 100s

📥 Batch 0001 salvo — 10 moedas — 2026-05-02T23:17:12.343911
⚠️  Rate limit (429) — aguardando 60s...
⚠️  Rate limit (429) — aguardando 60s...


---
## Parte 6 — Pipeline Spark Structured Streaming

O Spark lê os arquivos JSON do landing zone via `readStream`, aplica o schema definido no **Parte 3**, e escreve o resultado em Parquet via `writeStream`.

**Output Mode: `append`** — o mais eficiente para eventos novos: apenas os registros do micro-batch atual são gravados, sem precisar regerar toda a tabela.

**Trigger: `availableNow`** — processa todos os arquivos disponíveis agora e encerra. Ideal para demonstração. Em produção, usaríamos `processingTime='30 seconds'`.

In [0]:
# ── Leitura do stream de JSON ────────────────────────────────
# readStream com source 'JSON' monitora o diretório e processa novos arquivos
# O schema explícito evita inferência automática (cara em streaming)

# Schema completo incluindo campo de ingestão adicionado pelo produtor
FULL_SCHEMA = StructType(
    COIN_SCHEMA.fields + [
        StructField("ingestion_timestamp", StringType(), nullable=True)
    ]
)

raw_stream = (
    spark.readStream
    .format("json")
    .schema(FULL_SCHEMA)             # schema explícito na Parte 3
    .option("multiLine", True)       # cada arquivo contém um array JSON
    .option("mode", "DROPMALFORMED") # descarta registros malformados silenciosamente
    .load(LANDING_PATH)
)

print("✅ Stream de leitura configurado")
print(f"   Fonte:  {LANDING_PATH}")
print(f"   Formato: JSON (multiLine=True)")
print(f"   Modo de erros: DROPMALFORMED")
print(f"   Schema: {len(FULL_SCHEMA.fields)} campos")

✅ Stream de leitura configurado
   Fonte:  /Volumes/workspace/default/crypto_pipeline/landing
   Formato: JSON (multiLine=True)
   Modo de erros: DROPMALFORMED
   Schema: 12 campos


In [0]:
# ── Transformação: JSON → Parquet ────────────────────────────
# Transformações aplicadas antes da escrita:
#   1. Conversão do campo last_updated (string ISO) para TimestampType
#   2. Conversão do campo ingestion_timestamp para TimestampType
#   3. Cálculo do spread (diferença entre high_24h e low_24h)
#   4. Classificação da variação diária (alta / queda / estável)

transformed_stream = (
    raw_stream
    # ── Filtrar registros sem preço ou com preço zerado ──
    .filter(
        F.col("current_price").isNotNull() &
        (F.col("current_price") > 0) &
        F.col("id").isNotNull()
    )
    # ── Transformações de tipo e enriquecimento ───────────────────
    .withColumn(
        "event_time",
        F.to_timestamp(F.col("last_updated"), "yyyy-MM-dd'T'HH:mm:ss.SSS'Z'")
    )
    .withColumn(
        "ingestion_ts",
        F.to_timestamp(F.col("ingestion_timestamp"))
    )
    .withColumn(
        "spread_24h",
        F.round(F.col("high_24h") - F.col("low_24h"), 4)
    )
    .withColumn(
        "trend_24h",
        F.when(F.col("price_change_percentage_24h") >  1.0,  "alta")
         .when(F.col("price_change_percentage_24h") < -1.0,  "queda")
         .otherwise("estavel")
    )
    # ── Seleciona colunas finais para o Parquet ───────────────────
    .select(
        "id", "symbol", "name",
        "current_price", "market_cap", "market_cap_rank",
        "total_volume", "high_24h", "low_24h",
        "spread_24h",
        "price_change_percentage_24h",
        "trend_24h",
        "event_time",
        "ingestion_ts"
    )
)

print("✅ Transformações configuradas")
print("\nColunas no output final:")
for field in transformed_stream.schema.fields:
    print(f"  {field.name:<35} {str(field.dataType):<20}")

✅ Transformações configuradas

Colunas no output final:
  id                                  StringType()        
  symbol                              StringType()        
  name                                StringType()        
  current_price                       DoubleType()        
  market_cap                          DoubleType()        
  market_cap_rank                     LongType()          
  total_volume                        DoubleType()        
  high_24h                            DoubleType()        
  low_24h                             DoubleType()        
  spread_24h                          DoubleType()        
  price_change_percentage_24h         DoubleType()        
  trend_24h                           StringType()        
  event_time                          TimestampType()     
  ingestion_ts                        TimestampType()     


In [0]:
# ── Escrita do stream em Parquet (Delta Lake) ────────────────
# writeStream com:
#   - format: delta (Parquet com ACID transactions — padrão no Databricks)
#   - outputMode: append (apenas novos registros — mais eficiente)
#   - checkpointLocation: persiste estado do pipeline (permite retomada)
#   - trigger: availableNow → processa todos os arquivos disponíveis e para

stream_query = (
    transformed_stream
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .trigger(availableNow=True)      # para demonstração; em produção: processingTime='30 seconds'
    .start(OUTPUT_PATH)
)

# Aguarda o término do processamento
stream_query.awaitTermination()

print("\n✅ Stream processado com sucesso")
print(f"📦 Output salvo em: {OUTPUT_PATH} (formato Delta/Parquet)")


✅ Stream processado com sucesso
📦 Output salvo em: /Volumes/workspace/default/crypto_pipeline/output (formato Delta/Parquet)


---
## Parte 7 — Verificação do output base

Lemos o Parquet resultante para confirmar que o pipeline funcionou corretamente e que os dados estão no formato esperado.

In [0]:
# ── Leitura e verificação do output ──────────────────────────────────────────
output_df = spark.read.format("delta").load(OUTPUT_PATH)

total_records = output_df.count()
print(f"📊 Total de registros no output: {total_records}")
print(f"   Colunas: {len(output_df.columns)}")

print("\n📋 Amostra dos dados processados (5 registros):")
output_df.select(
    "symbol", "name", "current_price",
    "price_change_percentage_24h", "trend_24h", "event_time"
).show(5, truncate=False)

print("\n📈 Distribuição de tendências (24h):")
output_df.groupBy("trend_24h").count().orderBy("count", ascending=False).show()

📊 Total de registros no output: 390
   Colunas: 14

📋 Amostra dos dados processados (5 registros):
+------+--------+-------------+---------------------------+---------+-----------------------+
|symbol|name    |current_price|price_change_percentage_24h|trend_24h|event_time             |
+------+--------+-------------+---------------------------+---------+-----------------------+
|btc   |Bitcoin |78300.0      |-0.3282                    |estavel  |2026-05-02 14:17:52.351|
|eth   |Ethereum|2305.15      |-0.45705                   |estavel  |2026-05-02 14:17:52.524|
|usdt  |Tether  |0.999776     |-0.00212                   |estavel  |2026-05-02 14:17:52.118|
|xrp   |XRP     |1.39         |-0.34661                   |estavel  |2026-05-02 14:17:52.011|
|bnb   |BNB     |616.84       |-0.82177                   |estavel  |2026-05-02 14:17:52.053|
+------+--------+-------------+---------------------------+---------+-----------------------+
only showing top 5 rows

📈 Distribuição de tendências (

---
## Parte 8 — Agregação dos dados de streaming

Esta célula implementa a agregação diretamente sobre o **stream ativo**, não sobre um batch lido após o fato. O Spark gerencia o estado de forma incremental: a cada micro-batch, as estatísticas acumuladas são atualizadas sem reprocessar toda a tabela.

**Por que isso importa:** operar sobre `spark.read` (batch estático) após o stream ter terminado não é streaming — é ETL disfarçado. A versão correta usa `readStream → withWatermark → groupBy → writeStream`, com o engine gerenciando estado, retomada exata (exactly-once) e fechamento de janelas automaticamente.

**Watermark:** `withWatermark("event_time", "10 minutes")` define que eventos com atraso de até 10 minutos serão aceitos. Após esse limite, o Spark pode descartar o estado acumulado daquela janela, evitando crescimento de memória indefinido — o problema que o professor destacou na Aula 02.

**Output mode `update`:** emite apenas as linhas cujo estado mudou desde o último micro-batch. Mais eficiente que `complete` (reemite tudo) para agregações incrementais.

In [0]:
# ── Agregação em streaming real ─────────────────────────────────────
# Lê novamente o diretório de landing como stream (não batch)
# Isso garante que o Spark trate os dados como eventos contínuos

agg_stream_input = (
    spark.readStream
    .format("json")
    .schema(FULL_SCHEMA)
    .option("multiLine", True)
    .option("mode", "DROPMALFORMED")
    .load(LANDING_PATH)
    # Converte campos de tempo antes do watermark
    .withColumn(
        "event_time",
        F.to_timestamp(F.col("last_updated"), "yyyy-MM-dd'T'HH:mm:ss.SSS'Z'")
    )
    .withColumn(
        "ingestion_ts",
        F.to_timestamp(F.col("ingestion_timestamp"))
    )
    .filter(
        F.col("current_price").isNotNull() &
        (F.col("current_price") > 0) &
        F.col("id").isNotNull()
    )
    # trend_24h precisa ser calculado aqui pois agg_stream_input lê o JSON bruto,
    # que não possui esse campo — ele só existe em transformed_stream (Parte 6)
    .withColumn(
        "trend_24h",
        F.when(F.col("price_change_percentage_24h") >  1.0,  "alta")
         .when(F.col("price_change_percentage_24h") < -1.0,  "queda")
         .otherwise("estavel")
    )
    # ── Watermark: tolera eventos atrasados até 10 minutos ──────────────────
    # Sem watermark, o estado acumulado cresce indefinidamente em produção.
    # O Spark só pode descartar memória de estado quando sabe que nenhum
    # evento atrasado pode mais alterar aquela janela — o watermark é esse limite.
    .withWatermark("event_time", "10 minutes")
)

# Agrupamento e métricas por moeda
aggregated_stream = (
    agg_stream_input
    .groupBy("id", "symbol", "name")
    .agg(
        F.count("*")                                          .alias("snapshots"),
        F.avg("current_price")                               .alias("avg_price_usd"),
        F.max("current_price")                               .alias("max_price_usd"),
        F.min("current_price")                               .alias("min_price_usd"),
        F.last("current_price")                              .alias("last_price_usd"),
        F.avg("price_change_percentage_24h")                 .alias("avg_change_24h_pct"),
        F.avg("total_volume")                                .alias("avg_volume_usd"),
        F.avg("market_cap")                                  .alias("avg_mktcap_usd"),
        F.max("ingestion_ts")                                .alias("last_seen"),
        F.round(
            F.sum(F.when(F.col("trend_24h") == "alta", 1).otherwise(0)) /
            F.count("*") * 100, 1
        ).alias("pct_snapshots_alta")
    )
    .withColumn("avg_price_usd",      F.round("avg_price_usd",      4))
    .withColumn("max_price_usd",      F.round("max_price_usd",      4))
    .withColumn("min_price_usd",      F.round("min_price_usd",      4))
    .withColumn("last_price_usd",     F.round("last_price_usd",     4))
    .withColumn("avg_change_24h_pct", F.round("avg_change_24h_pct", 2))
)

# ── Escrita do stream de agregações ───────────────────────────────────────────
# outputMode "update": emite apenas linhas cujo estado mudou no micro-batch
# (mais eficiente que "complete", que reemite toda a tabela a cada batch)
agg_query = (
    aggregated_stream
    .writeStream
    .format("delta")
    .outputMode("complete")                         # só emite o que mudou
    .option("checkpointLocation", f"{BASE_PATH}/checkpoint_agg")
    .trigger(availableNow=True)
    .start(AGGS_PATH)
)

agg_query.awaitTermination()
print(f"\n✅ Agregações gravadas em streaming real em: {AGGS_PATH}")

# Leitura para verificação / resumo
aggregated_df = spark.read.format("delta").load(AGGS_PATH)
print(f"\n📊 Moedas agregadas: {aggregated_df.count()}")
aggregated_df.select(
    "symbol", "name", "snapshots",
    "avg_price_usd", "max_price_usd", "min_price_usd",
    "avg_change_24h_pct"
).show(10, truncate=False)


✅ Agregações gravadas em streaming real em: /Volumes/workspace/default/crypto_pipeline/aggregations

📊 Moedas agregadas: 10
+------+-----------------+---------+-------------+-------------+-------------+------------------+
|symbol|name             |snapshots|avg_price_usd|max_price_usd|min_price_usd|avg_change_24h_pct|
+------+-----------------+---------+-------------+-------------+-------------+------------------+
|steth |Lido Staked Ether|39       |2307.4546    |2322.32      |2302.1       |0.27              |
|doge  |Dogecoin         |39       |0.1089       |0.109        |0.1086       |-0.51             |
|eth   |Ethereum         |39       |2310.779     |2326.15      |2305.15      |0.32              |
|bnb   |BNB              |39       |617.4705     |619.74       |616.66       |-0.33             |
|ada   |Cardano          |39       |0.2497       |0.2514       |0.2488       |0.1               |
|btc   |Bitcoin          |39       |78466.4359   |78731.0      |78300.0      |0.3          

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8538790870813780>, line 77
     42 aggregated_stream = (
     43     agg_stream_input
     44     .groupBy("id", "symbol", "name")
   (...)
     64     .withColumn("avg_change_24h_pct", F.round("avg_change_24h_pct", 2))
     65 )
     67 # ── Escrita do stream de agregações ───────────────────────────────────────────
     68 # outputMode "update": emite apenas linhas cujo estado mudou no micro-batch
     69 # (mais eficiente que "complete", que reemite toda a tabela a cada batch)
     70 agg_query = (
     71     aggregated_stream
     72     .writeStream
     73     .format("delta")
     74     .outputMode("update")                         # só emite o que mudou
     75     .option("checkpointLocation", f"{BASE_PATH}/checkpoint_agg")
     76     .trigger(availableNow=True)
---> 77     .start(AGGS_PATH)
     78 )
     80 a

---
## Parte 9 — Window Functions — Janela deslizante (Sliding Window)

As janelas temporais são implementadas diretamente no **pipeline de streaming**, com watermark controlando o fechamento das janelas e liberação de estado.

**Watermark e window juntos:** o watermark (`10 minutes`) define quanto tempo o Spark aguarda por eventos atrasados antes de considerar uma janela fechada. Após esse limite, o resultado da janela é finalizado e o estado é descartado — exatamente o mecanismo descrito na Aula 02.

Duas janelas são calculadas em paralelo via dois `writeStream` separados:
- **Sliding (60min × 5min):** cada evento pertence a múltiplas janelas sobrepostas — padrão clássico de monitoramento financeiro ("preço médio das últimas 1h, atualizado a cada 5min")
- **Tumbling (30min):** blocos estanques sem sobreposição — equivale a uma barra OHLC de 30min

In [0]:
# ── Stream base reutilizado pelas duas janelas ──────────────────────
# Mesmo pipeline de entrada do readStream + watermark

window_stream_input = (
    spark.readStream
    .format("json")
    .schema(FULL_SCHEMA)
    .option("multiLine", True)
    .option("mode", "DROPMALFORMED")
    .load(LANDING_PATH)
    .withColumn(
        "event_time",
        F.to_timestamp(F.col("last_updated"), "yyyy-MM-dd'T'HH:mm:ss.SSS'Z'")
    )
    .withColumn(
        "spread_24h",
        F.round(F.col("high_24h") - F.col("low_24h"), 4)
    )
    .filter(
        F.col("current_price").isNotNull() &
        (F.col("current_price") > 0) &
        F.col("id").isNotNull()
    )
    # Watermark obrigatório para window functions em streaming
    # Define o atraso máximo tolerado antes de o Spark fechar a janela
    .withWatermark("event_time", "10 minutes")
)

# ── Janela deslizante (Sliding Window) ────────────────────────────────────────
# windowDuration=60min, slideDuration=5min
# Um mesmo evento aparece em múltiplas janelas sobrepostas
sliding_stream = (
    window_stream_input
    .groupBy(
        "symbol", "name",
        F.window(F.col("event_time"), "60 minutes", "5 minutes")
    )
    .agg(
        F.count("*")                   .alias("eventos_na_janela"),
        F.avg("current_price")         .alias("preco_medio_usd"),
        F.max("current_price")         .alias("preco_max_usd"),
        F.min("current_price")         .alias("preco_min_usd"),
        F.round(
            (F.max("current_price") - F.min("current_price")) /
            F.avg("current_price") * 100, 4
        )                              .alias("volatilidade_pct"),
        F.avg("total_volume")          .alias("volume_medio_usd")
    )
    .withColumn("preco_medio_usd", F.round("preco_medio_usd", 4))
    .withColumn("preco_max_usd",   F.round("preco_max_usd",   4))
    .withColumn("preco_min_usd",   F.round("preco_min_usd",   4))
    .withColumn("window_start",    F.col("window.start"))
    .withColumn("window_end",      F.col("window.end"))
    .drop("window")
)

WINDOWS_PATH = f"{BASE_PATH}/windows"

sliding_query = (
    sliding_stream
    .writeStream
    .format("delta")
    .outputMode("complete")          # janelas abertas são atualizadas a cada micro-batch
    .option("checkpointLocation", f"{BASE_PATH}/checkpoint_sliding")
    .trigger(availableNow=True)
    .start(f"{WINDOWS_PATH}/sliding")
)

sliding_query.awaitTermination()
print("✅ Sliding window processada em streaming real")

# Verificação
sliding_df = spark.read.format("delta").load(f"{WINDOWS_PATH}/sliding")
print(f"   Total de janelas deslizantes: {sliding_df.count()}")
sliding_df.select(
    "symbol", "window_start", "window_end",
    "eventos_na_janela", "preco_medio_usd", "volatilidade_pct"
).orderBy("symbol", "window_start").show(10, truncate=False)


✅ Sliding window processada em streaming real
   Total de janelas deslizantes: 370
+------+-------------------+-------------------+-----------------+---------------+----------------+
|symbol|window_start       |window_end         |eventos_na_janela|preco_medio_usd|volatilidade_pct|
+------+-------------------+-------------------+-----------------+---------------+----------------+
|ada   |2026-05-02 13:20:00|2026-05-02 14:20:00|5                |0.2488         |0.002           |
|ada   |2026-05-02 13:25:00|2026-05-02 14:25:00|5                |0.2488         |0.002           |
|ada   |2026-05-02 13:30:00|2026-05-02 14:30:00|5                |0.2488         |0.002           |
|ada   |2026-05-02 13:35:00|2026-05-02 14:35:00|9                |0.249          |0.1514          |
|ada   |2026-05-02 13:40:00|2026-05-02 14:40:00|24               |0.2493         |0.2632          |
|ada   |2026-05-02 13:45:00|2026-05-02 14:45:00|29               |0.2493         |0.2631          |
|ada   |2026-05-0

In [0]:
# ── Janela fixa (Tumbling Window) ────────────────────────────────────
# windowDuration=30min, sem slide — cada evento pertence a exatamente UMA janela
# Equivale a uma barra OHLC de 30 minutos no contexto de dados financeiros

tumbling_stream = (
    window_stream_input
    .groupBy(
        "symbol", "name",
        F.window(F.col("event_time"), "30 minutes")   # tumbling: só windowDuration
    )
    .agg(
        F.count("*")          .alias("eventos"),
        F.first("current_price").alias("preco_abertura_usd"),   # primeiro preço da janela
        F.last("current_price") .alias("preco_fechamento_usd"),  # último preço da janela
        F.max("current_price") .alias("preco_max_usd"),
        F.min("current_price") .alias("preco_min_usd"),
        F.avg("current_price") .alias("preco_medio_usd"),
        F.avg("total_volume")  .alias("volume_medio_usd")
    )
    .withColumn("preco_abertura_usd",   F.round("preco_abertura_usd",   4))
    .withColumn("preco_fechamento_usd", F.round("preco_fechamento_usd", 4))
    .withColumn("preco_max_usd",        F.round("preco_max_usd",        4))
    .withColumn("preco_min_usd",        F.round("preco_min_usd",        4))
    .withColumn("preco_medio_usd",      F.round("preco_medio_usd",      4))
    .withColumn("janela_inicio",        F.col("window.start"))
    .withColumn("janela_fim",           F.col("window.end"))
    .drop("window")
)

tumbling_query = (
    tumbling_stream
    .writeStream
    .format("delta")
    .outputMode("complete")
    .option("checkpointLocation", f"{BASE_PATH}/checkpoint_tumbling")
    .trigger(availableNow=True)
    .start(f"{WINDOWS_PATH}/tumbling")
)

tumbling_query.awaitTermination()
print("✅ Tumbling window processada em streaming real")

tumbling_df = spark.read.format("delta").load(f"{WINDOWS_PATH}/tumbling")
print(f"   Total de janelas fixas: {tumbling_df.count()}")
tumbling_df.select(
    "symbol", "janela_inicio", "janela_fim",
    "eventos", "preco_abertura_usd", "preco_fechamento_usd",
    "preco_max_usd", "preco_min_usd"
).orderBy("symbol", "janela_inicio").show(10, truncate=False)


✅ Tumbling window processada em streaming real
   Total de janelas fixas: 40
+------+-------------------+-------------------+-------+------------------+--------------------+-------------+-------------+
|symbol|janela_inicio      |janela_fim         |eventos|preco_abertura_usd|preco_fechamento_usd|preco_max_usd|preco_min_usd|
+------+-------------------+-------------------+-------+------------------+--------------------+-------------+-------------+
|ada   |2026-05-02 14:00:00|2026-05-02 14:30:00|5      |0.2488            |0.2488              |0.2488       |0.2488       |
|ada   |2026-05-02 14:30:00|2026-05-02 15:00:00|24     |0.2492            |0.2495              |0.2495       |0.2492       |
|ada   |2026-05-02 22:00:00|2026-05-02 22:30:00|5      |0.2514            |0.2513              |0.2514       |0.2513       |
|ada   |2026-05-02 22:30:00|2026-05-02 23:00:00|5      |0.2506            |0.2507              |0.2507       |0.2506       |
|bnb   |2026-05-02 14:00:00|2026-05-02 14:30:00|

---------------------------------------------------------------------------
StreamingQueryException                   Traceback (most recent call last)
File <command-5814022750998477>, line 40
      5 tumbling_stream = (
      6     window_stream_input
      7     .groupBy(
   (...)
     27     .drop("window")
     28 )
     30 tumbling_query = (
     31     tumbling_stream
     32     .writeStream
   (...)
     37     .start(f"{WINDOWS_PATH}/tumbling")
     38 )
---> 40 tumbling_query.awaitTermination()
     41 print("✅ Tumbling window processada em streaming real")
     43 tumbling_df = spark.read.format("delta").load(f"{WINDOWS_PATH}/tumbling")

File /databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/streaming/query.py:98, in StreamingQuery.awaitTermination(self, timeout)
     96 await_termination_cmd = pb2.StreamingQueryCommand.AwaitTerminationCommand()
     97 cmd.await_termination.CopyFrom(await_termination_cmd)
---> 98 self._execute_streaming_query_cmd(cmd)
    

---
## Parte 10 — Resumo executivo do pipeline

Consolida todas as métricas do pipeline em um relatório final, evidenciando o que foi entregue em cada etapa.

In [0]:
# ── Evidência 1: Schema do Delta Lake principal ───────────────────────────────
print("=" * 65)
print("  EVIDÊNCIA 1 — Schema do Delta Lake (output principal)")
print("=" * 65)
output_df = spark.read.format("delta").load(OUTPUT_PATH)
output_df.printSchema()
total_records = output_df.count()
print(f"  Total de registros: {total_records}")
print(f"  Colunas:           {len(output_df.columns)}")

# ── Evidência 2: Amostra dos dados transformados ──────────────────────────────
print("\n" + "=" * 65)
print("  EVIDÊNCIA 2 — Amostra dos dados transformados (5 registros)")
print("=" * 65)
output_df.select(
    "symbol", "name", "current_price",
    "spread_24h", "trend_24h", "event_time", "ingestion_ts"
).show(5, truncate=False)

# ── Evidência 3: Distribuição de tendências ───────────────────────────────────
print("=" * 65)
print("  EVIDÊNCIA 3 — Distribuição de tendências (24h)")
print("=" * 65)
output_df.groupBy("trend_24h").count().orderBy("count", ascending=False).show()

# ── Evidência 4: Resultado das agregações ───────────────────────────
print("=" * 65)
print("  EVIDÊNCIA 4 — Agregações por moeda")
print("=" * 65)
aggregated_df = spark.read.format("delta").load(AGGS_PATH)
aggregated_df.select(
    "symbol", "name", "snapshots",
    "avg_price_usd", "max_price_usd", "min_price_usd",
    "avg_change_24h_pct", "pct_snapshots_alta"
).orderBy("avg_mktcap_usd", ascending=False).show(10, truncate=False)

# ── Evidência 5: Janelas deslizantes ────────────────────────────────
print("=" * 65)
print("  EVIDÊNCIA 5 — Janelas deslizantes 60min×5min")
print("=" * 65)
sliding_df = spark.read.format("delta").load(f"{BASE_PATH}/windows/sliding")
sliding_df.select(
    "symbol", "window_start", "window_end",
    "eventos_na_janela", "preco_medio_usd", "volatilidade_pct"
).orderBy("symbol", "window_start").show(10, truncate=False)
print(f"  Total de janelas deslizantes calculadas: {sliding_df.count()}")

# ── Evidência 6: Janelas fixas (Bônus 3) ─────────────────────────────────────
print("=" * 65)
print("  EVIDÊNCIA 6 — Janelas fixas 30min / OHLC")
print("=" * 65)
tumbling_df = spark.read.format("delta").load(f"{BASE_PATH}/windows/tumbling")
tumbling_df.select(
    "symbol", "janela_inicio", "janela_fim",
    "preco_abertura_usd", "preco_fechamento_usd",
    "preco_max_usd", "preco_min_usd"
).orderBy("symbol", "janela_inicio").show(10, truncate=False)
print(f"  Total de janelas fixas calculadas: {tumbling_df.count()}")

# ── Evidência 7: Outputs gerados no DBFS ─────────────────────────────────────
print("\n" + "=" * 65)
print("  EVIDÊNCIA 7 — Arquivos gerados no DBFS (Unity Catalog Volume)")
print("=" * 65)
import subprocess
paths = [OUTPUT_PATH, AGGS_PATH,
         f"{BASE_PATH}/windows/sliding",
         f"{BASE_PATH}/windows/tumbling",
         f"{BASE_PATH}/checkpoint",
         f"{BASE_PATH}/checkpoint_agg",
         f"{BASE_PATH}/checkpoint_sliding",
         f"{BASE_PATH}/checkpoint_tumbling"]
for p in paths:
    try:
        files = dbutils.fs.ls(p)
        print(f"  {p}  →  {len(files)} arquivo(s)")
    except Exception:
        print(f"  {p}  →  (verifique no DBFS Explorer)")


  EVIDÊNCIA 1 — Schema do Delta Lake (output principal)
root
 |-- id: string (nullable = true)
 |-- symbol: string (nullable = true)
 |-- name: string (nullable = true)
 |-- current_price: double (nullable = true)
 |-- market_cap: double (nullable = true)
 |-- market_cap_rank: long (nullable = true)
 |-- total_volume: double (nullable = true)
 |-- high_24h: double (nullable = true)
 |-- low_24h: double (nullable = true)
 |-- spread_24h: double (nullable = true)
 |-- price_change_percentage_24h: double (nullable = true)
 |-- trend_24h: string (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- ingestion_ts: timestamp (nullable = true)

  Total de registros: 390
  Colunas:           14

  EVIDÊNCIA 2 — Amostra dos dados transformados (5 registros)
+------+--------+-------------+----------+---------+-----------------------+--------------------------+
|symbol|name    |current_price|spread_24h|trend_24h|event_time             |ingestion_ts              |
+------+--------+---

---
## Apêndice — Configuração de Deploy / CI-CD

Processo de deploy estruturado do pipeline. As configurações abaixo representam a camada de infraestrutura do trabalho.

### Estratégia de Deploy no Databricks

Em produção, o pipeline seria configurado como um **Databricks Job** executado periodicamente. O arquivo `databricks.yml` abaixo configura o deploy via Databricks Asset Bundles (DAB) — a abordagem moderna de "pipeline como código" no ecossistema Databricks.

```yaml
# databricks.yml — Configuração do pipeline como código (Databricks Asset Bundle)
bundle:
  name: stream_pipeline_coingecko

resources:
  jobs:
    crypto_stream_job:
      name: "CryptoStream - MBA Pipeline"
      description: "Pipeline de streaming de cotações de criptomoedas (CoinGecko API)"

      # Agendamento: executa a cada 15 minutos em produção
      schedule:
        quartz_cron_expression: "0 */15 * * * ?"
        timezone_id: "America/Sao_Paulo"

      # Configuração do cluster
      job_clusters:
        - job_cluster_key: streaming_cluster
          new_cluster:
            spark_version: "13.3.x-scala2.12"
            node_type_id: "Standard_DS3_v2"
            num_workers: 2
            spark_conf:
              spark.sql.streaming.checkpointLocation: "/dbfs/tmp/crypto_pipeline/checkpoint"
              spark.databricks.delta.autoCompact.enabled: "true"

      # Tarefas (Tasks) do job
      tasks:
        - task_key: ingest
          notebook_task:
            notebook_path: "notebooks/stream_pipeline_coingecko"
            base_parameters:
              NUM_BATCHES: "5"
              POLL_INTERVAL: "15"
          job_cluster_key: streaming_cluster
          libraries:
            - pypi:
                package: requests

      # Notificações em caso de falha
      email_notifications:
        on_failure:
          - engenharia@empresa.com.br
```

### GitHub Actions — CI/CD do Pipeline

```yaml
# .github/workflows/deploy_pipeline.yml
name: Deploy Stream Pipeline

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]

jobs:
  validate:
    name: Validar notebook
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3

      - name: Setup Python
        uses: actions/setup-python@v4
        with:
          python-version: "3.10"

      - name: Instalar dependências de validação
        run: |
          pip install nbformat nbconvert requests

      - name: Validar sintaxe do notebook
        run: |
          python -c "
          import nbformat
          nb = nbformat.read('notebooks/stream_pipeline_coingecko.ipynb', as_version=4)
          print(f'Notebook válido: {len(nb.cells)} células')
          "

      - name: Verificar conectividade com CoinGecko API
        run: |
          python -c "
          import requests
          r = requests.get('https://api.coingecko.com/api/v3/ping', timeout=10)
          assert r.status_code == 200, f'API indisponível: {r.status_code}'
          print('API disponível:', r.json())
          "

  deploy:
    name: Deploy no Databricks
    needs: validate
    runs-on: ubuntu-latest
    if: github.ref == 'refs/heads/main'
    steps:
      - uses: actions/checkout@v3

      - name: Instalar Databricks CLI
        run: pip install databricks-cli

      - name: Deploy via Databricks Asset Bundle
        env:
          DATABRICKS_HOST: ${{ secrets.DATABRICKS_HOST }}
          DATABRICKS_TOKEN: ${{ secrets.DATABRICKS_TOKEN }}
        run: |
          databricks bundle deploy --target prod
          echo "✅ Pipeline deployado em produção"
```